# Qwen3LLM Enable Thinking Demo

This notebook shows request-level Qwen3 thinking control with `enable_thinking`.

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd()
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from llm_utils import Qwen3LLM

CLIENT = "http://localhost:8001/v1"
MODEL = "/tmp/scratch/models/Qwen3.5-122B-A10B"

## Shared Prompt

Both calls use the same messages. Only `enable_thinking` changes.

In [2]:
messages = [
    {"role": "system", "content": "Answer clearly and put the final answer in one sentence."},
    {"role": "user", "content": "What is 17 * 23?"},
]

## Thinking Disabled

This sends `extra_body.chat_template_kwargs.enable_thinking = False`.

In [3]:
llm = Qwen3LLM(
    client=CLIENT,
    model=MODEL,
    enable_thinking=False,
    max_tokens=256,
)

message = llm(messages)
print(message.content or "")

17 multiplied by 23 equals 391.


## Thinking Enabled

This sends `extra_body.chat_template_kwargs.enable_thinking = True` and includes a reasoning budget.

In [4]:
llm = Qwen3LLM(
    client=CLIENT,
    model=MODEL,
    enable_thinking=True,
    thinking_max_tokens=1024,
    content_max_tokens=256,
)

message = llm(messages)
print("Reasoning:")
print(getattr(message, "reasoning", None) or "")
print("\nAnswer:")
print(message.content or "")

Reasoning:
Thinking Process:

1.  **Analyze the Request:** The user is asking for the product of 17 and 23. They want a clear answer and the final answer in one sentence.

2.  **Calculate the Product:**
    *   $17 \times 23$
    *   Break it down: $17 \times (20 + 3)$
    *   $17 \times 20 = 340$
    *   $17 \times 3 = 51$
    *   $340 + 51 = 391$

    *   Alternatively, standard multiplication:
        ```
          17
        x 23
        ----
          51  (3 * 17)
         340  (20 * 17)
        ----
         391
        ```

3.  **Formulate the Output:** The user requested a clear answer with the final answer in one sentence.

4.  **Draft the Sentence:** "17 multiplied by 23 is 391."

5.  **Review Constraints:**
    *   Clear? Yes.
    *   Final answer in one sentence? Yes.

6.  **Final Output Generation:** 17 multiplied by 23 equals 391.cw


Answer:


17 multiplied by 23 equals 391.


In [5]:
llm.inspect_history()

## Per-Call Override

A default thinking setting can be overridden per request.

In [6]:
llm = Qwen3LLM(client=CLIENT, model=MODEL, enable_thinking=True)

message = llm(
    messages,
    enable_thinking=False,
    content_max_tokens=256,
)
print(message.content or "")

17 multiplied by 23 equals 391.
